In [10]:
import os
import time
import requests
from typing import Dict, Any, List, Optional
from datetime import datetime
import pandas as pd

In [11]:
# ======================
# Config
# ======================
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "GobJlP7vKjn4Z5Jw")
ROOT = "https://ensembledata.com/apis"
POSTS_ENDPOINT = "/instagram/user/posts"

# ======================
# Helpers
# ======================
def safe_get(d: Any, path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

In [13]:
def fetch_user_posts_page(
    user_id: int,
    depth: int = 1,
    chunk_size: int = 5,
    oldest_timestamp: Optional[int] = None,
    start_cursor: str = "",
    alternative_method: bool = False,
    timeout: int = 60
) -> Dict[str, Any]:
    """PAID CALL: fetch one page (chunk) of posts for a user."""
    params = {
        "user_id": int(user_id),
        "depth": depth,
        "chunk_size": int(chunk_size),
        "start_cursor": start_cursor,
        "alternative_method": alternative_method,
        "token": TOKEN,
    }
    if oldest_timestamp is not None:
        params["oldest_timestamp"] = int(oldest_timestamp)

    r = requests.get(ROOT + POSTS_ENDPOINT, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()


In [14]:

def parse_posts_to_rows(posts_data: Dict[str, Any], brand_label: str = "") -> List[Dict[str, Any]]:
    rows = []
    posts = posts_data.get("data", {}).get("posts", [])

    for post in posts:
        node = post.get("node", {}) or {}

        owner = node.get("owner", {}) or {}
        owner_id = owner.get("id", "")
        owner_username = owner.get("username", "")

        post_id = node.get("id", "")
        ts = node.get("taken_at_timestamp", None)
        post_type = node.get("__typename", "")
        shortcode = node.get("shortcode", "")

        readable_date = ""
        if isinstance(ts, (int, float)) and ts:
            readable_date = datetime.fromtimestamp(int(ts)).strftime("%Y-%m-%d %H:%M:%S")

        # ---- cover: use thumbnail_src + thumbnail_resources (exact)
        cover_url = node.get("thumbnail_src", "")
        cover_width, cover_height = None, None
        thumbs = node.get("thumbnail_resources", [])
        if isinstance(thumbs, list) and thumbs:
            best_thumb = max(thumbs, key=lambda r: r.get("config_width", 0))
            cover_width = best_thumb.get("config_width")
            cover_height = best_thumb.get("config_height")

        likes = safe_get(node, ["edge_media_preview_like", "count"], 0) or 0
        comments = safe_get(node, ["edge_media_to_comment", "count"], 0) or 0
        is_video = bool(node.get("is_video", False))
        video_views = node.get("video_view_count", None) if is_video else None

        caption_edges = safe_get(node, ["edge_media_to_caption", "edges"], []) or []
        full_caption = safe_get(caption_edges[0], ["node", "text"], "") if caption_edges else ""
        accessibility_text = node.get("accessibility_caption", "") or ""

        coauthors = node.get("coauthor_producers", []) or []
        cocreators = ", ".join([c.get("username", "") for c in coauthors if isinstance(c, dict)]) if coauthors else ""

        is_paid = bool(node.get("is_paid_partnership", False))
        is_affiliate = bool(node.get("is_affiliate", False))
        sponsored = "Paid Partnership" if is_paid else ("Affiliate" if is_affiliate else "")

        # ---- carousel
        image_count = 0
        carousel_media_urls = ""

        if post_type == "GraphSidecar":
            child_edges = safe_get(node, ["edge_sidecar_to_children", "edges"], []) or []
            urls = []
            for e in child_edges:
                child = safe_get(e, ["node"], {}) or {}
                child_is_video = bool(child.get("is_video", False))
                if not child_is_video:
                    image_count += 1
                u = child.get("display_url", "")
                if u:
                    urls.append(u)
            carousel_media_urls = ", ".join(urls)
        else:
            image_count = 0 if is_video else (1 if node.get("display_url") else 0)

        rows.append({
            "brand": brand_label,
            "ig_brand_ID": owner_id,
            "brand_Username": owner_username,

            "Post ID": post_id,
            "Timestamp": readable_date,
            "Post Type": post_type,
            "Shortcode": shortcode,

            "Cover url": cover_url,
            "Cover width": cover_width,
            "Cover height": cover_height,

            "Likes": likes,
            "Comments": comments,
            "Video Views": (video_views if video_views is not None else ""),

            "Image Count": image_count,
            "Carousel Media URLs": carousel_media_urls,

            "Full Caption": full_caption,
            "Accessibility Text": accessibility_text,

            "Co-creators": cocreators,
            "Sponsored/Partnership": sponsored,
        })

    return rows

In [15]:
def scrape_account_posts(
    user_id: int,
    brand_label: str = "",
    chunk_size: int = 3,
    depth: int = 1,
    oldest_timestamp: Optional[int] = None,
    max_pages: Optional[int] = 1,   # ✅ set 1 for “only 5 posts”; set None for “all posts”
    sleep_seconds: float = 0.2
) -> List[Dict[str, Any]]:
    """
    Scrape posts page-by-page using last_cursor pagination.
    Returns a list of rows (one row per post).
    """
    all_rows: List[Dict[str, Any]] = []
    cursor = ""  # start from most recent
    page = 0

    while True:
        page += 1
        posts_data = fetch_user_posts_page(
            user_id=user_id,
            depth=depth,
            chunk_size=chunk_size,
            oldest_timestamp=oldest_timestamp,
            start_cursor=cursor
        )

        all_rows.extend(parse_posts_to_rows(posts_data, brand_label=brand_label))

        next_cursor = posts_data.get("data", {}).get("last_cursor", "") or ""
        got_posts = len(posts_data.get("data", {}).get("posts", []))

        print(f"  page={page} posts={got_posts} next_cursor={'YES' if next_cursor else 'NO'}")

        # Stop conditions
        if got_posts == 0:
            break
        if not next_cursor or next_cursor == cursor:
            break
        if max_pages is not None and page >= max_pages:
            break

        cursor = next_cursor
        time.sleep(sleep_seconds)

    return all_rows

In [16]:
def run_scrape_from_csv(
    accounts_csv: str,
    out_csv: str = "instagram_posts_all.csv",
    chunk_size: int = 5,
    depth: int = 1,
    oldest_timestamp: Optional[int] = None,
    max_pages: Optional[int] = 1,     # ✅ 1=only 5 posts, None=all posts
    sleep_seconds: float = 0.2
) -> pd.DataFrame:
    accounts = pd.read_csv(accounts_csv)
    if "user_id" not in accounts.columns:
        raise ValueError("CSV must contain a 'user_id' column (Instagram numeric id).")

    all_rows: List[Dict[str, Any]] = []

    for i, row in accounts.iterrows():
        user_id = int(row["user_id"])
        brand_label = str(row["brand"]) if "brand" in accounts.columns else ""

        print(f"[{i+1}/{len(accounts)}] user_id={user_id} brand={brand_label}")

        try:
            rows = scrape_account_posts(
                user_id=user_id,
                brand_label=brand_label,
                chunk_size=chunk_size,
                depth=depth,
                oldest_timestamp=oldest_timestamp,
                max_pages=max_pages,
                sleep_seconds=sleep_seconds
            )
            all_rows.extend(rows)
            print(f"  -> total rows added: {len(rows)}")
        except Exception as e:
            print(f"  -> ERROR: {e}")

    df = pd.DataFrame(all_rows)
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"\nSaved: {out_csv} | rows={len(df)}")
    return df


if __name__ == "__main__":
    # ✅ For now: only 5 posts per account
    # Set max_pages=1
    df = run_scrape_from_csv(
        accounts_csv="accounts.csv",
        out_csv="instagram_posts_test.csv",
        chunk_size=5,
        depth=1,
        oldest_timestamp=None,
        max_pages=1,          # ✅ change to None when you want ALL posts
        sleep_seconds=0.2
    )

    # ✅ Later for ALL posts:
    # df = run_scrape_from_csv(..., max_pages=None)


[1/1] user_id=20269764 brand=adidas
  page=1 posts=5 next_cursor=YES
  -> total rows added: 5

Saved: instagram_posts_test.csv | rows=5


In [17]:
df

,brand,ig_brand_ID,brand_Username,Post ID,Timestamp,Post Type,Shortcode,Cover url,Cover width,Cover height,Likes,Comments,Video Views,Image Count,Carousel Media URLs,Full Caption,Accessibility Text,Co-creators,Sponsored/Partnership
0,adidas,20269764,adidas,3809410779849992311,2026-01-13 15:46:14,GraphImage,DTdvuvQgKh3,https://instagram.fyei4-1.fna.fbcdn.net/v/t51....,640,640,17304,118,,1,,Mikaela Shiffrin takes another night slalom wi...,,"adidasus, mikaelashiffrin, adidasterrex",
1,adidas,193981203,adidasusfootball,3809251738705497264,2026-01-13 10:30:00,GraphSidecar,DTdLkYplIyw,https://instagram.fyei1-2.fna.fbcdn.net/v/t51....,640,640,27883,184,,3,https://instagram.fyei1-2.fna.fbcdn.net/v/t51....,Designed by the best. For the best. 🛸 @romeodunze,,"romeodunze, adidas",
2,adidas,193981203,adidasusfootball,3809206644870750465,2026-01-13 09:00:00,GraphVideo,DTdBULvlCEB,https://instagram.fyei1-2.fna.fbcdn.net/v/t51....,640,640,69441,343,321704,0,,Introducing adizero Horizon 🛸\nMade to create ...,,adidas,
3,adidas,20269764,adidas,3805588212811591105,2026-01-08 09:13:03,GraphVideo,DTQKlESDG3B,https://instagram.fyei4-1.fna.fbcdn.net/v/t51....,640,640,43176,356,445240,0,,"Strip back the excess. Lift, jump, run. Dropse...",,"haileyvanlith, antonelaroccuzzo, emilio_sakraya_",
4,adidas,20269764,adidas,3797754994313148895,2025-12-28 13:48:16,GraphImage,DS0VgvEADnf,https://instagram.fyei4-1.fna.fbcdn.net/v/t51....,640,640,14517,144,,1,,@mikaelashiffrin delivers another masterclass ...,,"adidasus, mikaelashiffrin, adidasterrex",
